# BreastCare: test and export ianpan/mammoscreen

Needs **GPU T4**, **Internet on**, and the dataset **awsaf49/cbis-ddsm-breast-cancer-image-dataset** attached.
Scores the CBIS-DDSM test patients with the mammoscreen ensemble (and smaller combinations of its nets),
exports the nets to ONNX for the BreastCare backend, and writes test metrics and Java test fixtures.
Download `mammoscreen_model.zip` from the Output panel afterwards.

In [ ]:
import os, pathlib
WORK = pathlib.Path('/kaggle/working/ML-Service')
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

In [ ]:
%%writefile metrics.py
"""Classification metrics for benign (0) / malignant (1) scores, shared by train.py and evaluate.py."""
import numpy as np
from sklearn.metrics import roc_auc_score


def at_threshold(labels, scores, threshold):
    """Accuracy, precision, recall (sensitivity), specificity, F1, ROC-AUC and the confusion counts."""
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    predicted = (scores >= threshold).astype(int)
    tp = int(((predicted == 1) & (labels == 1)).sum())
    tn = int(((predicted == 0) & (labels == 0)).sum())
    fp = int(((predicted == 1) & (labels == 0)).sum())
    fn = int(((predicted == 0) & (labels == 1)).sum())

    def ratio(a, b):
        return a / b if b else 0.0

    precision = ratio(tp, tp + fp)
    recall = ratio(tp, tp + fn)
    return {
        "images": int(len(labels)),
        "malignant": int(labels.sum()),
        "benign": int(len(labels) - labels.sum()),
        "accuracy": ratio(tp + tn, len(labels)),
        "precision": precision,
        "recall": recall,  # sensitivity: share of malignant images the model flags
        "specificity": ratio(tn, tn + fp),  # share of benign images the model clears
        "f1": ratio(2 * precision * recall, precision + recall),
        "roc_auc": float(roc_auc_score(labels, scores)) if 0 < labels.sum() < len(labels) else None,
        "confusion_matrix": {"true_positive": tp, "false_positive": fp, "true_negative": tn, "false_negative": fn},
    }


def best_threshold(labels, scores):
    """The threshold with the highest balanced accuracy (Youden's J = sensitivity + specificity - 1).

    Chosen on the validation set only, never on the test set, so the test numbers stay honest.
    """
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    best, best_j = 0.5, -1.0
    for threshold in np.unique(np.round(scores, 4)):
        m = at_threshold(labels, scores, threshold)
        j = m["recall"] + m["specificity"] - 1
        if j > best_j:
            best, best_j = float(threshold), j
    return best


def patient_bootstrap(labels, scores, patients, threshold, rounds=1000, seed=0):
    """95% intervals for accuracy, recall, specificity and AUC, resampling whole patients.

    A patient's CC and MLO images are not independent, so resampling images would make the intervals too narrow.
    """
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    patients = np.asarray(patients)
    groups = [np.flatnonzero(patients == p) for p in np.unique(patients)]
    rng = np.random.default_rng(seed)
    keys = ("accuracy", "recall", "specificity", "roc_auc")
    samples = {k: [] for k in keys}
    for _ in range(rounds):
        index = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        m = at_threshold(labels[index], scores[index], threshold)
        for k in keys:
            if m[k] is not None:
                samples[k].append(m[k])
    return {k: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] for k, v in samples.items() if v}


def report(labels, scores, views, patients, threshold, bootstrap=True):
    """Overall metrics plus the same metrics for CC and MLO images separately."""
    labels, scores, views = np.asarray(labels), np.asarray(scores), np.asarray(views)
    result = {"threshold": threshold, "overall": at_threshold(labels, scores, threshold), "by_view": {}}
    if bootstrap:
        result["overall"]["ci95"] = patient_bootstrap(labels, scores, patients, threshold)
    for view in ("CC", "MLO"):
        mask = views == view
        if mask.any():
            result["by_view"][view] = at_threshold(labels[mask], scores[mask], threshold)
            if bootstrap:
                result["by_view"][view]["ci95"] = patient_bootstrap(
                    labels[mask], scores[mask], np.asarray(patients)[mask], threshold)
    result["patients"] = int(len(set(patients)))
    return result

In [ ]:
%%writefile mammoscreen_pipeline.py
"""Preprocessing for the ianpan/mammoscreen model, written so the Java backend can repeat it exactly.

Model:   https://huggingface.co/ianpan/mammoscreen (Apache-2.0), commit b619a889a8214f3b3f9e7427c2603608a44e6d0a.
         Ensemble of 3 EfficientNetV2-S nets for screening mammography (cancer probability), trained on
         CBIS-DDSM then the RSNA Screening Mammography challenge. Author's RSNA holdout AUC: 0.945.
Cropper: https://huggingface.co/ianpan/mammo-crop (Apache-2.0), commit bec2e302d47176d743a86d276d8402f876ad0f9f.

Steps (same as the model card, without albumentations, so nothing depends on a library version):
  1. 8-bit grayscale image.
  2. Crop to the breast: resize to 256 x 256 (OpenCV INTER_LINEAR), the crop net returns normalised x, y, w, h.
  3. For each net: nets 0 and 1 pad the width with black to their aspect ratio, then resize to their size;
     net 2 scales the longest side to 1536 and pads the rest with black, centred.
  4. Each net takes float pixels 0-255 (shape 1 x 1 x H x W; normalisation is inside the exported graph) and
     returns P(cancer). The result is the mean over the nets used.
"""
import cv2
import numpy as np

CROP_SIZE = 256
NETS = [
    {"name": "net0", "height": 2048, "width": 1024, "pad_to_aspect": True},
    {"name": "net1", "height": 1920, "width": 1280, "pad_to_aspect": True},
    {"name": "net2", "height": 1536, "width": 1536, "pad_to_aspect": False},
]
MIN_CROP_FRACTION = 0.1  # a crop smaller than this share of the image is treated as a failed crop


def resize_linear(img, height, width):
    return cv2.resize(img, (width, height), interpolation=cv2.INTER_LINEAR)


def crop_input(gray):
    """1 x 1 x 256 x 256 float32 for the crop net."""
    return resize_linear(gray, CROP_SIZE, CROP_SIZE).astype(np.float32)[None, None]


def crop_box(coords, height, width):
    """Pixel box from the crop net's normalised x, y, w, h (as the model card's .int() rescaling)."""
    x = int(coords[0] * width)
    y = int(coords[1] * height)
    w = int(coords[2] * width)
    h = int(coords[3] * height)
    x, y = max(0, min(x, width - 1)), max(0, min(y, height - 1))
    w, h = min(w, width - x), min(h, height - y)
    if w < MIN_CROP_FRACTION * width or h < MIN_CROP_FRACTION * height:
        return 0, 0, width, height
    return x, y, w, h


def pad_to_aspect_ratio(img, aspect_ratio):
    h, w = img.shape[:2]
    if h / w > aspect_ratio:
        new_w = round(h / aspect_ratio)
        left = (new_w - w) // 2
        img = np.pad(img, ((0, 0), (left, new_w - w - left)), mode="constant", constant_values=0)
    return img


def longest_max_size(img, max_size):
    h, w = img.shape[:2]
    scale = max_size / float(max(h, w))
    return resize_linear(img, round(h * scale), round(w * scale))


def pad_if_needed(img, height, width):
    h, w = img.shape[:2]
    top, left = max(0, (height - h) // 2), max(0, (width - w) // 2)
    return np.pad(img, ((top, max(0, height - h - top)), (left, max(0, width - w - left))),
                  mode="constant", constant_values=0)


def net_input(cropped, spec):
    """1 x 1 x H x W float32 (0-255) for one ensemble net."""
    if spec["pad_to_aspect"]:
        img = pad_to_aspect_ratio(cropped, spec["height"] / spec["width"])
        img = resize_linear(img, spec["height"], spec["width"])
    else:
        img = pad_if_needed(longest_max_size(cropped, spec["height"]), spec["height"], spec["width"])
    return img.astype(np.float32)[None, None]

In [ ]:
%%writefile evaluate_mammoscreen.py
"""Test ianpan/mammoscreen on CBIS-DDSM test patients and export it to ONNX for the backend. Runs on Kaggle (GPU).

Run inside kaggle_mammoscreen.ipynb (built by make_mammoscreen_notebook.py), with the Kaggle dataset
awsaf49/cbis-ddsm-breast-cancer-image-dataset attached (full-resolution CBIS-DDSM JPEGs, original aspect ratio).

Writes to weights/mammoscreen/:
  crop.onnx, net0.onnx, net1.onnx, net2.onnx   exported models (input 1x1xHxW float 0-255, output P(cancer))
  mammoscreen_metrics.json                     test results for each combination of nets, overall/CC/MLO, and
                                               per breast (CC+MLO averaged), with 95% intervals
  fixtures/                                    images and expected outputs for the Java parity tests

Test set: the official CBIS-DDSM test split minus patients who also appear in the training split (the same rule
as prepare_cbis.py). Labels: MALIGNANT -> 1, BENIGN / BENIGN_WITHOUT_CALLBACK -> 0. The decision threshold for
each combination is chosen on a sample of CBIS-DDSM training images, never on the test images.

Caveat recorded in the metrics: mammoscreen was pretrained on CBIS-DDSM, and its card does not say whether the
test split was held out, so these CBIS numbers may be optimistic. Its own RSNA holdout AUC is 0.945.
"""
import csv
import glob
import json
import os
import random
import re
import time
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import torch

import metrics
from mammoscreen_pipeline import NETS, crop_box, crop_input, net_input

SCREEN_REPO, SCREEN_REV = "ianpan/mammoscreen", "b619a889a8214f3b3f9e7427c2603608a44e6d0a"
CROP_REPO, CROP_REV = "ianpan/mammo-crop", "bec2e302d47176d743a86d276d8402f876ad0f9f"
OUT = Path("weights/mammoscreen")
COMBOS = {"ensemble3": ["net0", "net1", "net2"], "net0+net1": ["net0", "net1"], "net0": ["net0"]}
THRESHOLD_SAMPLE = 400
KEY = re.compile(r"^(Mass|Calc)-(Training|Test)_(P_\d+)_(LEFT|RIGHT)_(CC|MLO)$")


def find_dataset_root():
    for candidate in glob.glob("/kaggle/input/**/csv/dicom_info.csv", recursive=True):
        return Path(candidate).parent.parent
    raise SystemExit("Attach the Kaggle dataset awsaf49/cbis-ddsm-breast-cancer-image-dataset")


def load_cases(root):
    """One record per full mammogram, found through the series UID in the case CSVs' image file path.

    (The dataset's dicom_info.csv leaves SeriesDescription blank for most Calc-Test images, so it cannot be
    used to find full mammograms.)
    """
    files = {}
    with open(root / "csv" / "dicom_info.csv", newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            files[row["SeriesInstanceUID"]] = root / "jpeg" / Path(row["image_path"]).relative_to("CBIS-DDSM/jpeg")
    images = {}
    for path in sorted(glob.glob(str(root / "csv" / "*_case_description_*_set.csv"))):
        with open(path, newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                folder, _study, series = row["image file path"].strip().split("/")[:3]
                match = KEY.match(folder)
                if not match or series not in files:
                    raise SystemExit(f"cannot find the full image for {folder}")
                kind, split, patient, side, view = match.groups()
                label = 1 if row["pathology"] == "MALIGNANT" else 0
                if folder in images:
                    images[folder]["label"] = max(images[folder]["label"], label)
                    continue
                images[folder] = {"key": folder, "path": str(files[series]), "patient_id": patient, "side": side,
                                  "view": view, "kind": kind, "split": split, "label": label}
    return list(images.values())


class CropNet(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):  # 1x1x256x256, 0-255 -> normalised xywh
        return self.model(x)


class CancerNet(torch.nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, x):  # 1x1xHxW, 0-255 -> P(cancer), shape (1, 1)
        x = self.net.normalize(x)
        features = self.net.pooling(self.net.backbone(x)).flatten(1)
        return self.net.linear(features)[:, :1].sigmoid()


def read_gray(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise RuntimeError(f"cannot read {path}")
    return img


@torch.inference_mode()
def score(image_path, crop_net, nets, device):
    gray = read_gray(image_path)
    coords = crop_net(torch.from_numpy(crop_input(gray)).to(device))[0].float().cpu().numpy()
    x, y, w, h = crop_box(coords, *gray.shape[:2])
    cropped = gray[y:y + h, x:x + w]
    return {spec["name"]: float(nets[spec["name"]](torch.from_numpy(net_input(cropped, spec)).to(device))[0, 0])
            for spec in NETS}


def combo_scores(per_net, names):
    return np.array([np.mean([s[n] for n in names]) for s in per_net])


def breast_level(images, scores):
    groups = defaultdict(list)
    for image, s in zip(images, scores):
        groups[(image["patient_id"], image["side"], image["kind"])].append((image, s))
    labels, values, patients = [], [], []
    for (patient, _, _), members in groups.items():
        if {m[0]["view"] for m in members} == {"CC", "MLO"}:
            labels.append(max(m[0]["label"] for m in members))
            values.append(float(np.mean([m[1] for m in members])))
            patients.append(patient)
    return labels, values, patients


def main():
    from transformers import AutoModel

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    OUT.mkdir(parents=True, exist_ok=True)
    root = find_dataset_root()
    images = load_cases(root)
    train_patients = {i["patient_id"] for i in images if i["split"] == "Training"}
    test = [i for i in images if i["split"] == "Test" and i["patient_id"] not in train_patients]
    sample = random.Random(7).sample([i for i in images if i["split"] == "Training"], THRESHOLD_SAMPLE)
    print(f"test {len(test)} images ({sum(i['label'] for i in test)} malignant), threshold sample {len(sample)}")

    crop_model = AutoModel.from_pretrained(CROP_REPO, revision=CROP_REV, trust_remote_code=True).eval()
    screen = AutoModel.from_pretrained(SCREEN_REPO, revision=SCREEN_REV, trust_remote_code=True).eval()
    crop_net = CropNet(crop_model).to(device).eval()
    nets = {spec["name"]: CancerNet(getattr(screen, spec["name"])).to(device).eval() for spec in NETS}

    # 1. Our preprocessing must give the same answer as the model's own (albumentations) code.
    check, check_error = [], None
    try:
        for image in test[:8]:
            gray = read_gray(image["path"])
            with torch.inference_mode():
                coords = crop_model(torch.from_numpy(crop_model.preprocess(gray)).float()[None, None].to(device),
                                    torch.tensor([gray.shape[:2]]).to(device))[0].cpu().numpy()
                x, y, w, h = coords
                official = float(screen({"img": gray[y:y + h, x:x + w]}, device=str(device))["cancer"][0, 0])
            ours = float(np.mean(list(score(image["path"], crop_net, nets, device).values())))
            check.append(abs(official - ours))
        print(f"preprocessing check: largest difference from the model card's code {max(check):.5f}")
    except Exception as e:  # the card's code depends on albumentations' API; our own pipeline does not
        check_error = f"{type(e).__name__}: {e}"
        print("preprocessing check could not run:", check_error)

    # 2. Score every image with every net.
    started = time.time()
    test_scores = [score(i["path"], crop_net, nets, device) for i in test]
    sample_scores = [score(i["path"], crop_net, nets, device) for i in sample]
    print(f"scored {len(test) + len(sample)} images in {time.time() - started:.0f}s")

    results = {}
    labels = [i["label"] for i in test]
    for combo, names in COMBOS.items():
        threshold = metrics.best_threshold([i["label"] for i in sample], combo_scores(sample_scores, names))
        values = combo_scores(test_scores, names)
        report = metrics.report(labels, values, [i["view"] for i in test], [i["patient_id"] for i in test], threshold)
        b_labels, b_values, b_patients = breast_level(test, values)
        report["per_breast_cc_mlo"] = {
            **metrics.at_threshold(b_labels, b_values, threshold),
            "ci95": metrics.patient_bootstrap(b_labels, b_values, b_patients, threshold),
        }
        results[combo] = {"nets": names, **report}
        o = report["overall"]
        print(f"{combo:10s} auc {o['roc_auc']:.3f} recall {o['recall']:.3f} spec {o['specificity']:.3f} "
              f"acc {o['accuracy']:.3f} | per breast auc {report['per_breast_cc_mlo']['roc_auc']:.3f}")

    # 3. Export to ONNX and check it matches PyTorch.
    parity = {}
    exports = [("crop", crop_net, (1, 1, 256, 256))] + [
        (spec["name"], nets[spec["name"]], (1, 1, spec["height"], spec["width"])) for spec in NETS]
    import onnxruntime as ort
    for name, module, shape in exports:
        module = module.float().cpu().eval()
        example = torch.rand(*shape) * 255
        path = OUT / f"{name}.onnx"
        torch.onnx.export(module, example, path, input_names=["image"], output_names=["output"], dynamo=False,
                          opset_version=17)
        session = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
        with torch.inference_mode():
            expected = module(example).numpy()
        parity[name] = float(np.abs(session.run(None, {"image": example.numpy()})[0] - expected).max())
    print("onnx vs torch", parity)
    assert max(parity.values()) < 1e-3, "ONNX export does not match PyTorch"

    # 4. Fixtures for the Java tests: a smaller copy of a real test mammogram, and what each step must produce.
    fixtures = OUT / "fixtures"
    fixtures.mkdir(exist_ok=True)
    picked = next(i for i in test if i["label"] == 1)
    gray = read_gray(picked["path"])
    small = cv2.resize(gray, (gray.shape[1] // 4, gray.shape[0] // 4), interpolation=cv2.INTER_AREA)
    cv2.imwrite(str(fixtures / "mammo.png"), small)
    cv2.imwrite(str(fixtures / "mammo_crop256.png"), cv2.resize(small, (256, 256), interpolation=cv2.INTER_LINEAR))
    sessions = {name: ort.InferenceSession(str(OUT / f"{name}.onnx"), providers=["CPUExecutionProvider"])
                for name, _, _ in exports}
    from mammoscreen_pipeline import crop_input as ci, crop_box as cb, net_input as ni
    coords = sessions["crop"].run(None, {"image": ci(small)})[0][0]
    box = cb(coords, *small.shape[:2])
    cropped = small[box[1]:box[1] + box[3], box[0]:box[0] + box[2]]
    expected = {spec["name"]: float(sessions[spec["name"]].run(None, {"image": ni(cropped, spec)})[0][0, 0]) for spec in NETS}
    cv2.imwrite(str(fixtures / "mammo_net0_input.png"), ni(cropped, NETS[0])[0, 0].astype(np.uint8))
    cv2.imwrite(str(fixtures / "mammo_net2_input.png"), ni(cropped, NETS[2])[0, 0].astype(np.uint8))
    (fixtures / "expected.json").write_text(json.dumps({"crop_coords": coords.tolist(), "crop_box": list(box),
                                                        "probabilities": expected}, indent=2))

    # 5. CPU speed of one image (Kaggle CPU; the laptop is measured separately).
    timing = {}
    for name, _, shape in exports:
        x = (np.random.rand(*shape) * 255).astype(np.float32)
        sessions[name].run(None, {"image": x})
        t = time.time()
        for _ in range(3):
            sessions[name].run(None, {"image": x})
        timing[name] = round((time.time() - t) / 3, 3)
    print("kaggle cpu seconds per image", timing)

    (OUT / "mammoscreen_metrics.json").write_text(json.dumps({
        "model": {"repo": SCREEN_REPO, "revision": SCREEN_REV, "crop_repo": CROP_REPO, "crop_revision": CROP_REV,
                  "licence": "Apache-2.0", "author_rsna_holdout_auc": 0.945},
        "evaluated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "test_set": {
            "name": "CBIS-DDSM full mammograms, official test split",
            "source": "TCIA CBIS-DDSM (CC BY 3.0) via Kaggle awsaf49/cbis-ddsm-breast-cancer-image-dataset",
            "patient_disjoint": True,
            "note": "Patients also in the training split removed. Every image has a biopsy-proven benign or "
                    "malignant finding. mammoscreen was pretrained on CBIS-DDSM and its card does not say whether "
                    "this test split was held out, so these numbers may be optimistic.",
        },
        "threshold_sample": f"{THRESHOLD_SAMPLE} CBIS-DDSM training images",
        "preprocessing_max_difference": max(check) if check else None,
        "preprocessing_check_error": check_error,
        "onnx_parity": parity,
        "kaggle_cpu_seconds": timing,
        "combinations": results,
    }, indent=2))
    print(f"wrote {OUT}")


if __name__ == "__main__":
    main()

In [ ]:
# The model's own code needs the transformers version it was saved with (newer 5.x breaks custom models).
!pip install -q "transformers==4.47.0" "onnx>=1.16" "onnxruntime>=1.18" timm albumentations
import torch, timm, transformers, cv2
print(torch.__version__, timm.__version__, transformers.__version__, cv2.__version__)
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4"

In [ ]:
!python evaluate_mammoscreen.py

In [ ]:
import shutil, zipfile
out = WORK / 'weights' / 'mammoscreen'
with zipfile.ZipFile('/kaggle/working/mammoscreen_model.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for path in out.rglob('*'):
        if path.is_file():
            z.write(path, path.relative_to(out))
shutil.rmtree(WORK / 'weights')
print('done: download mammoscreen_model.zip')